# 유형 04 — 그리디 (Greedy)

> **대상**: 완전탐색까지 익힌 단계.
> **목표**: "매 순간 가장 좋아 보이는 선택을 하면 전체 답이 된다"는 **그리디가 통하는 상황을 알아보는** 눈 기르기.

그리디는 **"지금 당장 최선을 고르고 뒤돌아보지 않는"** 전략입니다. 완전탐색이 모든 경우를 다 본다면, 그리디는 매 단계에서 한 길만 고릅니다 — 빠르지만, **아무 문제에나 통하지 않습니다.**


## 0) 핵심 주의 — 그리디는 "항상" 맞지 않는다

그리디의 가장 큰 함정: **그럴듯해 보여도 틀릴 수 있다.**

전형적인 반례 — **동전 거스름돈**:
- 동전이 `[1, 5, 10]`이면 큰 것부터 욕심내면 항상 최소 개수 (그리디 OK)
- 동전이 `[1, 3, 4]`이고 6원을 거슬러야 하면? 그리디는 `4+1+1=3개`, 하지만 정답은 `3+3=2개`. **그리디 실패.**

> 규칙: 그리디를 떠올렸으면 **작은 반례를 직접 만들어 검산**하세요. 안 깨지면 그리디, 깨지면 DP(유형 07)나 완전탐색으로 갑니다.

**그리디가 통하는 두 신호**:
1. **정렬했을 때** 앞에서부터 차례로 고르면 되는 구조 (회의실, 일정)
2. "큰 것부터" 또는 "급한 것부터" 처리하면 손해가 없는 구조 (거스름돈류, 단 위 반례 주의)

## 1) 기본 패턴 — 정렬 후 순서대로 선택

그리디의 절반은 **"무엇을 기준으로 정렬하느냐"**입니다 (유형 02와 연결).

```python
# "회의가 가장 빨리 끝나는 것부터 고르면 최대 개수를 넣을 수 있다" (회의실 배정)
def max_meetings(meetings):
    # 끝나는 시간 기준 정렬 ← 그리디의 핵심 선택
    meetings.sort(key=lambda x: x[1])
    count = 0
    end = 0
    for start, finish in meetings:
        if start >= end:        # 직전 회의가 끝난 뒤 시작 가능하면
            count += 1
            end = finish        # 선택하고 끝 시간 갱신
    return count

max_meetings([(1, 4), (3, 5), (0, 6), (5, 7), (5, 9), (8, 9)])  # 3
```

> 직관: 끝을 빨리 비울수록 뒤에 더 많이 넣을 수 있다 → "끝나는 시간"이 정렬 기준. **왜 이 기준이 손해가 없는지** 설명할 수 있으면 그리디가 맞습니다.

In [7]:
def max_meetings(meetings):
    meetings.sort(key=lambda x: x[1])
    count = 0
    end = 0
    for start, finish in meetings:
        if start >= end:
            count += 1
            end = finish
    return count

print(max_meetings([(1, 4), (3, 5), (0, 6), (5, 7), (5, 9), (8, 9)]))

3


## 2) 기본 패턴 — 큰 것/급한 것부터

```python
# "거스름돈을 동전 [1, 5, 10, 50, 100, 500]로 최소 개수로" (그리디 OK인 화폐 체계)
def change(amount):
    coins = [500, 100, 50, 10, 5, 1]
    count = 0
    for coin in coins:          # 큰 동전부터
        count += amount // coin # 최대한 많이 쓰고
        amount %= coin          # 나머지로 넘어감
    return count

change(1260)  # 6  (500 + 500 + 100 + 100 + 50 + 10)
```

> 이 화폐 체계는 "큰 단위가 작은 단위의 배수"라 그리디가 항상 최적입니다. 0번 섹션의 `[1,3,4]` 반례와 비교해 **왜 여기선 안전한지** 짚어보세요.

In [4]:
def change(amount):
    coins = [500, 100, 50, 10, 5, 1]
    count = 0 
    for coin in coins:
        count += amount // coin
        amount %= coin
    return count

change(1260)

6

## 대표 문제 풀이 — 체육복 (프로그래머스 42862)

> 학생 `n`명. 체육복을 도난당한 학생 `lost`, 여벌이 있는 학생 `reserve`. 여벌이 있으면 자신 또는 **바로 앞뒤 번호** 학생에게 빌려줄 수 있다. 체육 수업을 들을 수 있는 최대 학생 수는?

**① 신호**: "바로 앞뒤", "최대한 많이" — 가까운 사람부터 빌려주면 손해 없음
**② 패턴**: 그리디 (번호 순서대로, 앞 학생부터 빌려줌)

```python
def solution(n, lost, reserve):
    # 여벌이 있는데 도난도 당한 학생은 자기 걸 입음 → 후보에서 제외
    real_lost = set(lost) - set(reserve)
    real_reserve = set(reserve) - set(lost)

    answer = n - len(real_lost)
    for r in sorted(real_reserve):      # 번호 작은 여벌부터
        if r - 1 in real_lost:          # 앞 번호 먼저 빌려주기
            real_lost.remove(r - 1)
            answer += 1
        elif r + 1 in real_lost:        # 없으면 뒤 번호
            real_lost.remove(r + 1)
            answer += 1
    return answer
```

**검증:**

```python
print(solution(5, [2, 4], [1, 3, 5]))   # 5  (1→2, 3 or 5→4)
print(solution(5, [2, 4], [3]))         # 4  (3이 2 또는 4 한 명만 도움)
print(solution(3, [3], [1]))            # 2  (멀어서 못 빌려줌)

assert solution(5, [2, 4], [1, 3, 5]) == 5
assert solution(5, [2, 4], [3]) == 4
assert solution(3, [3], [1]) == 2
print("통과 ✅")
```

> 가르칠 포인트:
> 1. **여벌+도난 동시 학생 처리** — `set` 차집합으로 "실제로 빌려줄 수 있는 사람 / 빌려야 하는 사람"을 먼저 정리. 이걸 놓치면 틀립니다.
> 2. **앞 번호부터(`r-1` 먼저)** — 정렬해서 작은 번호부터, 앞을 먼저 빌려주는 게 그리디 선택. 뒤(`r+1`)를 먼저 주면 그 뒤 학생이 못 받는 경우가 생겨 손해.

In [5]:
def solution(n, lost, reverse):
    real_lost = set(lost)- set(reverse)
    real_reverse = set(reverse) - set(lost)

    answer = n - len(real_lost)
    for r in sorted(real_reverse):
        if r - 1 in real_lost:
            real_lost.remove(r - 1)
            answer += 1
        elif r + 1 in real_lost:
            real_lost.remove(r + 1)
            answer += 1
    return answer

In [6]:
print(solution(5, [2, 4], [1, 3, 5]))   # 5  (1→2, 3 or 5→4)
print(solution(5, [2, 4], [3]))         # 4  (3이 2 또는 4 한 명만 도움)
print(solution(3, [3], [1]))            # 2  (멀어서 못 빌려줌)

assert solution(5, [2, 4], [1, 3, 5]) == 5
assert solution(5, [2, 4], [3]) == 4
assert solution(3, [3], [1]) == 2
print("통과 ✅")

5
4
2
통과 ✅


## 직접 풀어보기 (연습)

힌트를 보고 `TODO`를 채운 뒤 검증 셀을 실행하세요. 그리디는 **"어떤 기준으로 정렬/선택하면 손해가 없을까?"**를 먼저 정합니다. 막히면 작은 반례로 검산!

### 연습 1 — 최소 동전 개수 (그리디 OK 화폐)

> 금액 `amount`를 동전 `[1, 10, 50, 100, 500]`으로 거슬러줄 때 최소 동전 개수를 구하라.
> **힌트**: 큰 동전부터 `amount // coin` 만큼 쓰고 `amount %= coin`.

```python
def solution(amount):
    # TODO: 직접 작성
    pass

# 검증
assert solution(660) == 4    # 500 + 100 + 50 + 10
assert solution(1) == 1
assert solution(0) == 0
print("통과 ✅")
```

### 연습 2 — 구명보트 (프로그래머스 42885)

> 사람 무게 리스트 `people`, 보트 한 척에 최대 2명이고 무게 합이 `limit` 이하. 모든 사람을 옮길 최소 보트 수는?
> **힌트(투 포인터 그리디)**: 정렬 후 **가장 무거운 사람과 가장 가벼운 사람을 짝**지어 보낸다. 둘이 같이 못 타면 무거운 사람 혼자.

```python
def solution(people, limit):
    # TODO: people 정렬 후 양 끝 포인터로 직접 작성
    pass

# 검증
assert solution([70, 50, 80, 50], 100) == 3   # (50,50),(80),(70)
assert solution([70, 80, 50], 100) == 3
assert solution([40, 60, 90], 100) == 2       # (40,60),(90)
print("통과 ✅")
```

### 연습 3 — 큰 수 만들기 (도전)

> 숫자 문자열 `number`에서 `k`개를 제거해 만들 수 있는 **가장 큰 수**를 문자열로 반환하라. (예: `"1924"`, `k=2` → `"94"`)
> **힌트(스택 그리디)**: 왼쪽부터 보며, 지금 숫자가 스택 맨 위보다 크면 위를 빼고(아직 제거 횟수가 남았으면) 넣는다. 01의 스택 + 그리디 결합.

```python
def solution(number, k):
    # TODO: 스택으로 직접 작성 (제거 횟수 k를 다 쓰는 것에 주의)
    pass

# 검증
assert solution("1924", 2) == "94"
assert solution("1231234", 3) == "3234"
assert solution("4177252841", 4) == "775841"
print("통과 ✅")
```